# RT-DETRv2: query-based detector on every fifth frame

We train RT-DETRv2 with a ResNet-50-vd backbone from the official COCO checkpoint
`PekingU/rtdetr_v2_r50vd` under the protocol of the Faster R-CNN B1 run, so that the two
detector families see the same data and the same number of image exposures.

| Setting | Faster R-CNN B1 | RT-DETRv2 runs |
| --- | --- | --- |
| Training frames | 4,868, stride 5 within each clip | The same 4,868 frames |
| Seed, flips, epochs, batch size | 2026, 0.5, 5, 2 | The same |
| Image exposures | 24,340 | The same |
| Resize | Shorter side 640, longer side at most 1024 | The same, padded to a multiple of 32 |
| Backbone | ResNet-50 FPN | ResNet-50-vd with a hybrid encoder |
| Optimizer | SGD 0.0025 | AdamW 1e-4, backbone 1e-5, gradient clipping 0.1 |
| Schedule | 100 warmup updates, x0.1 at 2/3 and 90% | 500 warmup updates, x0.1 at 2/3 and 90% |
| Ignored regions | Excluded from background sampling | Unmatched queries inside them get no classification loss |
| Post-processing | Class-wise NMS at 0.5, 500 detections | Top 500 (query, class) pairs, no NMS |
| Validation after every epoch | COCO maxDets 100 and 500 | The same |
| Reported checkpoint | Final epoch | Final epoch |

The experiments run in this order, each changing one setting of the baseline:
`baseline_q300`, `nwd_loss`, `queries_500`, and `denoising_300`.
The recipe and the experiment tables are in `config/rt_detr_baseline.toml`.

**Stages.** Set `HM2_STAGE` (or `STAGE` in section 2) to `"probe"` for a short timing run,
`"full"` for training, or `"analysis"` for the comparison with the CNN detectors after
training. The analysis stage needs no GPU.

## 1. Runtime

We record the actual GPU and software versions. The hosted runtime keeps its own
PyTorch/TorchVision pair; we install the Transformers version that the code was checked
against when it differs, because the loss reads the RT-DETRv2 output fields directly.

In [ ]:
import gc
import hashlib
import importlib
import importlib.metadata
import importlib.util
import json
import logging
import platform
import shutil
import subprocess
import sys
import time
import urllib.request
from pathlib import Path
from zipfile import ZIP_DEFLATED, ZipFile

if sys.version_info < (3, 12):
    raise RuntimeError("Use a Python 3.12 or newer runtime.")

KAGGLE_INPUT = Path("/kaggle/input")
IS_KAGGLE = KAGGLE_INPUT.is_dir()
TRANSFORMERS_VERSION = "5.22.2"
requirements = {
    "polars": "polars>=1.31",
    "plotly": "plotly>=6",
    "pycocotools": "pycocotools>=2.0.11",
    "decouple": "python-decouple>=3.8",
    "PIL": "Pillow>=11",
    "tqdm": "tqdm>=4.67",
    "ipywidgets": "ipywidgets>=8",
    "scipy": "scipy>=1.11",
    "googleapiclient": "google-api-python-client",
    "google.oauth2": "google-auth",
}
missing = [
    package for module, package in requirements.items() if importlib.util.find_spec(module) is None
]
installed_transformers = (
    importlib.metadata.version("transformers")
    if importlib.util.find_spec("transformers") is not None
    else None
)
if installed_transformers != TRANSFORMERS_VERSION:
    missing.append(f"transformers=={TRANSFORMERS_VERSION}")
if missing and IS_KAGGLE:
    uv = shutil.which("uv")
    if uv is None:
        installer = Path("/kaggle/working/install_uv.sh")
        urllib.request.urlretrieve("https://astral.sh/uv/install.sh", installer)
        subprocess.run(["sh", str(installer)], check=True)
        uv = str(Path.home() / ".local/bin/uv")
    subprocess.run(
        [uv, "pip", "install", "--system", "--python", sys.executable, *missing], check=True
    )

import polars as pl
import torch
import torchvision
from IPython.display import display

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    datefmt="%Y-%m-%dT%H:%M:%S",
    stream=sys.stdout,
    force=True,
)
LOGGER = logging.getLogger("rt_detr_kaggle")
torch.set_num_threads(2)
torch.backends.cudnn.benchmark = False
pl.from_dicts(
    [
        {
            "Python": platform.python_version(),
            "PyTorch": torch.__version__,
            "TorchVision": torchvision.__version__,
            "Transformers": importlib.metadata.version("transformers"),
            "CUDA": torch.version.cuda,
            "GPU": torch.cuda.get_device_name() if torch.cuda.is_available() else "None",
            "Free VRAM GiB": (
                torch.cuda.mem_get_info()[0] / 1024**3 if torch.cuda.is_available() else 0.0
            ),
        }
    ]
)

## 2. Code, stage and output paths

We load the project code from `hm2_retinanet/inputs/rt_detr_project.zip` on Drive through
the `HM2_DRIVE_TOKEN` secret, verify its checksum and read the attached competition data in
place. Outputs go to `runs/rt_detr/<batch id>/<experiment>/`, and durable snapshots to
`hm2_retinanet/batches/<batch id>/<experiment>/` on Drive. `HM2_EXPERIMENTS` may list a
comma-separated subset of the experiments, for example `nwd_loss`.

In [ ]:
if not IS_KAGGLE:
    raise RuntimeError("Run this notebook on Kaggle; it trains and evaluates detectors.")

from google.oauth2.credentials import Credentials
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

UserSecretsClient = importlib.import_module("kaggle_secrets").UserSecretsClient
token = json.loads(UserSecretsClient().get_secret("HM2_DRIVE_TOKEN"))
if not token.get("refresh_token"):
    raise ValueError("HM2_DRIVE_TOKEN must contain refreshable Drive credentials.")
credentials = Credentials.from_authorized_user_info(token)
del token
service = build("drive", "v3", credentials=credentials, cache_discovery=False)
drive_root = "16wH9gNp13kfa4T039nsiFcZwYOlEJI5Z"
root_files = (
    service.files()
    .list(
        q=f"'{drive_root}' in parents and trashed = false",
        fields="files(id,name)",
        pageSize=1000,
    )
    .execute(num_retries=3)["files"]
)
inputs = [entry for entry in root_files if entry["name"] == "inputs"]
if len(inputs) != 1:
    raise ValueError("Expected one inputs folder in hm2_retinanet on Drive.")
input_files = (
    service.files()
    .list(
        q=f"'{inputs[0]['id']}' in parents and trashed = false",
        fields="files(id,name,size,md5Checksum)",
        pageSize=1000,
    )
    .execute(num_retries=3)["files"]
)
projects = [entry for entry in input_files if entry["name"] == "rt_detr_project.zip"]
if len(projects) != 1:
    raise FileNotFoundError("Upload exactly one rt_detr_project.zip to hm2_retinanet/inputs.")
project = projects[0]
archive_path = Path("/kaggle/temp/rt_detr_project.zip")
archive_path.parent.mkdir(parents=True, exist_ok=True)
temporary = archive_path.with_suffix(".incoming")
with temporary.open("wb") as stream:
    transfer = MediaIoBaseDownload(stream, service.files().get_media(fileId=project["id"]))
    has_completed = False
    while not has_completed:
        _, has_completed = transfer.next_chunk(num_retries=3)
with temporary.open("rb") as stream:
    actual_md5 = hashlib.file_digest(stream, "md5").hexdigest()
if actual_md5 != project["md5Checksum"] or temporary.stat().st_size != int(project["size"]):
    raise ValueError("Project archive transfer checksum mismatch.")
temporary.replace(archive_path)
REPOSITORY_ROOT = Path("/kaggle/temp/hm2_rt_detr")
if REPOSITORY_ROOT.exists():
    shutil.rmtree(REPOSITORY_ROOT)
REPOSITORY_ROOT.mkdir(parents=True)
with ZipFile(archive_path) as archive:
    required_paths = {
        "src/hm2/rt_detr.py",
        "src/hm2/rt_detr_experiments.py",
        "src/hm2/rt_detr_analysis.py",
        "src/hm2/faster_rcnn_experiments.py",
        "src/hm2/drive_api.py",
        "config/rt_detr_baseline.toml",
    }
    if not required_paths.issubset(archive.namelist()):
        raise ValueError("The Drive archive is missing RT-DETR code or configuration.")
    for entry in archive.infolist():
        if (
            not (REPOSITORY_ROOT / entry.filename)
            .resolve()
            .is_relative_to(REPOSITORY_ROOT.resolve())
        ):
            raise ValueError("Project archive contains an unsafe path.")
    archive.extractall(REPOSITORY_ROOT)
LOGGER.info("Downloaded and verified project code from Drive: %s", archive_path.name)
sys.path.insert(0, str(REPOSITORY_ROOT / "src"))

from decouple import config

from hm2.drive_api import (
    DriveApi,
    restore_checkpoint_from_drive_api,
    sync_checkpoint_to_drive_api,
    sync_results_to_drive_api,
    verify_drive_api_checkpoint,
)
from hm2.experiment_checkpoint import publish_snapshot, write_json
from hm2.faster_rcnn_experiments import (
    compute_source_identity,
    export_test_submission,
    find_competition_root,
    find_previous_output,
    measure_inference_speed,
    prepare_experiment_tables,
    restore_previous_output,
    select_experiment_frames,
    select_probe_frames,
    stage_images,
    summarize_experiment,
    verify_selection,
)
from hm2.rt_detr_analysis import run_comparison_analysis
from hm2.rt_detr_experiments import (
    RECORD_FILES,
    RUN_ORDER,
    build_rt_detr_experiment_config,
    download_committed_files,
    experiment_label,
    find_drive_folder,
    list_experiments,
    load_final_rt_detr_model,
    load_rt_detr_settings,
    run_rt_detr_experiment,
    run_rt_detr_timing_probe,
)
from hm2.tiny_subset import flight_prefix

STAGE = "full"
STAGE = config("HM2_STAGE", default=STAGE)
if STAGE not in {"probe", "full", "analysis"}:
    raise ValueError('STAGE must be "probe", "full" or "analysis".')
if STAGE != "analysis" and not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime for the probe and full stages.")
CONFIG_PATH = REPOSITORY_ROOT / "config/rt_detr_baseline.toml"
ALL_EXPERIMENTS = list_experiments(CONFIG_PATH)
EXPERIMENTS = [
    name.strip()
    for name in config("HM2_EXPERIMENTS", default=",".join(RUN_ORDER)).split(",")
    if name.strip()
]
PROBE_EXPERIMENT = config("HM2_PROBE_EXPERIMENT", default="baseline_q300")
unknown = set(EXPERIMENTS + [PROBE_EXPERIMENT]) - set(ALL_EXPERIMENTS)
if unknown:
    raise ValueError(f"Unknown experiments: {sorted(unknown)}; choose from {ALL_EXPERIMENTS}")
BATCH_ID = "rt_detr_v2_r50_stride5_5epochs_seed2026"
SESSION_HOURS = 10.5
LOCAL_BATCH = Path("/kaggle/working/runs/rt_detr") / BATCH_ID
PROBE_DIRECTORY = LOCAL_BATCH / "_timing_probe"
ANALYSIS_DIRECTORY = LOCAL_BATCH / "_analysis"
STAGING_ROOT = Path("/kaggle/temp/rt_detr_images")
SETTINGS = {name: load_rt_detr_settings(CONFIG_PATH, name) for name in ALL_EXPERIMENTS}
BASE_SETTINGS = SETTINGS["baseline_q300"]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LOCAL_BATCH.mkdir(parents=True, exist_ok=True)
{
    "Stage": STAGE,
    "Batch": BATCH_ID,
    "Experiments": EXPERIMENTS if STAGE == "full" else ALL_EXPERIMENTS,
    "Probe experiment": PROBE_EXPERIMENT,
    "Local outputs": str(LOCAL_BATCH),
    "Session hours": SESSION_HOURS,
}

## 3. Dataset staging

We build the frame, target and ignored-region tables with the same table builder as the
B1 run and copy only the images that the stage needs to fast local storage. The analysis
stage reads only the validation annotations.

In [ ]:
SOURCE_ROOT = find_competition_root(KAGGLE_INPUT)
frames, objects, ignored = prepare_experiment_tables(
    SOURCE_ROOT, Path("/kaggle/working/data/metadata")
)
selected, validation = select_experiment_frames(frames, BASE_SETTINGS["dataset"])
test_frames = frames.filter(pl.col("split") == "test")
IMAGE_ROOT = SOURCE_ROOT
staging = None
if STAGE != "analysis" and BASE_SETTINGS["dataset"]["should_stage_images"]:
    if STAGE == "probe":
        probe_frames = select_probe_frames(selected, validation, ignored, BASE_SETTINGS)
        needed = [probe_frames["train"], probe_frames["ignore"], probe_frames["validation"]]
    else:
        needed = [selected, validation, test_frames]
    IMAGE_ROOT = STAGING_ROOT
    staging = stage_images(
        SOURCE_ROOT, IMAGE_ROOT, [path for part in needed for path in part["relative_path"]]
    )
pl.DataFrame(
    {
        "Item": ["Image root", "Staged files", "Copied megabytes", "Copy seconds"],
        "Value": [
            str(IMAGE_ROOT),
            str(staging["files"]) if staging else "0",
            f"{staging['copied_megabytes']:.0f}" if staging else "0",
            f"{staging['seconds']:.0f}" if staging else "0",
        ],
    }
)

## 4. Reproducibility, Drive and recipe checks

We verify the training selection against the expected counts and the flight separation,
identify the annotation files and the selection by hash, and check Drive access. Before
training we also run the shared save, upload, download and restore probe. The table lists
what each experiment changes; every other setting equals the baseline.

In [ ]:
verify_selection(selected, validation, BASE_SETTINGS["dataset"])
selected_keys = selected["image_key"].to_list()
SOURCE_IDENTITY = compute_source_identity(SOURCE_ROOT, selected_keys)
api = DriveApi(service)
batch_folder = api.folder(api.folder(drive_root, "batches"), BATCH_ID)
cache = Path("/kaggle/temp/drive_cache") / BATCH_ID
quota = service.about().get(fields="storageQuota").execute(num_retries=3)["storageQuota"]
if "limit" in quota and int(quota["limit"]) - int(quota.get("usage", 0)) < 2 * 1024**3:
    raise OSError("Keep at least 2 GiB free on Drive for checkpoint replacement.")
if STAGE == "full":
    probe_result = verify_drive_api_checkpoint(
        api, cache / "probe", api.folder(batch_folder, "_probe_api")
    )
    LOGGER.info("Drive checkpoint save and restore verified: %s", probe_result)
pl.from_dicts(
    [
        {
            "Experiment": experiment_label(name),
            "Queries": SETTINGS[name]["model"]["num_queries"],
            "Denoising queries": SETTINGS[name]["model"]["num_denoising"],
            "Box loss": SETTINGS[name]["loss"]["box_loss"],
            "Backbone LR": SETTINGS[name]["training"]["backbone_learning_rate"],
            "Transformer LR": SETTINGS[name]["training"]["learning_rate"],
            "Ignore-aware": SETTINGS[name]["model"]["is_ignore_aware"],
            "Training frames": selected.height,
            "Training flights": len({flight_prefix(clip) for clip in selected["clip"]}),
            "Validation frames": validation.height,
        }
        for name in ALL_EXPERIMENTS
    ]
)

## 5. Timing probe

We train a few updates of one experiment with the real recipe and time inference. The probe
also checks four conventions on the GPU: the anchors follow each input size instead of the
checkpoint's fixed 640 by 640, our loss equals the Hugging Face loss on one shared forward
pass, matching keeps every object of the frame with the most ignored regions, and the
ignore rule reports how many queries it drops there. It also measures the object size in
model pixels next to the NWD constant. Nothing is checkpointed; projections are linear
extrapolations, not results.

In [ ]:
if STAGE == "probe":
    probe = run_rt_detr_timing_probe(
        IMAGE_ROOT,
        SOURCE_ROOT,
        frames,
        objects,
        ignored,
        PROBE_DIRECTORY,
        SETTINGS[PROBE_EXPERIMENT],
    )
    api.upload(PROBE_DIRECTORY / "probe.json", api.folder(batch_folder, "_timing_probe"))
    batch_size = SETTINGS[PROBE_EXPERIMENT]["evaluation"]["batch_size"]
    ignore_check, sizes = probe["ignore_check"], probe["object_size_model_pixels"]
    rows = {
        "Experiment": experiment_label(PROBE_EXPERIMENT),
        "Device": probe["gpu"] or probe["device"],
        "Transformers": probe["transformers"],
        "Checkpoint commit": str(probe["model_commit"]),
        "Anchor size and embedding size": f"{probe['anchor_image_size']}, {probe['eval_size']}",
        "Loss relative difference to Hugging Face": (
            f"{probe['reference_loss_check']['relative_difference']:.2e}"
        ),
        "Training images per second": f"{probe['train_images_per_second']:.2f}",
        "Seconds per update": f"{probe['train_seconds_per_update']:.2f}",
        "Peak GPU MiB": f"{probe['peak_gpu_mib']:.0f}",
        "Inference ms per frame, batch 1": (
            f"{probe['inference']['batch_1']['milliseconds_per_frame']:.1f}"
        ),
        f"Inference ms per frame, batch {batch_size}": (
            f"{probe['inference'][f'batch_{batch_size}']['milliseconds_per_frame']:.1f}"
        ),
        "COCO evaluation seconds per frame": f"{probe['evaluation_seconds_per_frame']:.4f}",
        "Detections per frame": f"{probe['detections_per_frame_mean']:.0f}",
        "Ignore frame: objects, matched queries": (
            f"{ignore_check['objects']}, {ignore_check['matched_queries']:.0f}"
        ),
        "Ignore frame: regions, dropped queries (final layer)": (
            f"{ignore_check['ignored_regions']}, {ignore_check['excluded_queries_final']:.0f}"
        ),
        "Object size in model pixels, median and mean": (
            f"{sizes['median']:.1f}, {sizes['mean']:.1f}"
        ),
        "NWD constant": f"{probe['nwd_constant']:.1f}",
        "Projected training minutes": f"{probe['projection']['train_minutes']:.0f}",
        "Projected validation minutes": f"{probe['projection']['validation_minutes']:.0f}",
        "Projected total minutes": f"{probe['projection']['total_minutes']:.0f}",
    }
    if staging and staging["megabytes_per_second"]:
        full_files = selected.height + validation.height + test_frames.height
        minutes = full_files * staging["megabytes"] / staging["files"]
        rows["Projected image copy minutes"] = (
            f"{minutes / staging['megabytes_per_second'] / 60:.1f}"
        )
    display(pl.DataFrame({"Measurement": list(rows), "Value": list(rows.values())}))

## 6. Training and validation

We run the selected experiments in order. Each one first restores its last verified Drive
snapshot, so a completed experiment is skipped and an interrupted one resumes. The model
after the fifth epoch is the result; the per-epoch validation is recorded as information
only. When the session budget runs out, the run pauses after a checkpoint and the next
session continues it. After an experiment completes, we time inference on 300 validation
frames at batch size 1, write an unsubmitted test file and synchronize everything to Drive.

In [ ]:
statuses: dict = {}
if STAGE == "full":
    session_deadline = time.monotonic() + SESSION_HOURS * 3600
    for name in EXPERIMENTS:
        remaining = session_deadline - time.monotonic()
        if remaining < 15 * 60:
            LOGGER.info("Session budget reached before %s. Rerun to continue.", name)
            break
        settings = SETTINGS[name]
        output = LOCAL_BATCH / name
        run_folder = api.folder(batch_folder, name)
        experiment_config = build_rt_detr_experiment_config(
            settings, name, selected_keys, SOURCE_IDENTITY
        )
        previous = find_previous_output(KAGGLE_INPUT, BATCH_ID, name, family="rt_detr")
        if previous is not None and restore_previous_output(previous, output):
            LOGGER.info("Restored the checkpoint of an attached previous version: %s", previous)
        progress = restore_checkpoint_from_drive_api(
            api, run_folder, output, experiment_config, cache / name
        )
        LOGGER.info(
            "%s: restored epoch=%s cursor=%s updates=%s",
            name,
            progress.get("epochs_completed", 0),
            progress.get("batch_cursor", 0),
            progress.get("global_step", 0),
        )

        def persist_checkpoint(path: Path, folder_id: str = run_folder) -> None:
            sync_checkpoint_to_drive_api(api, path, folder_id)

        status = run_rt_detr_experiment(
            IMAGE_ROOT,
            SOURCE_ROOT,
            frames,
            objects,
            ignored,
            output,
            settings,
            name,
            seconds=remaining,
            on_checkpoint=persist_checkpoint,
        )
        statuses[name] = status
        gc.collect()
        torch.cuda.empty_cache()
        if status["is_stopped"]:
            raise InterruptedError(f"Remove {output / 'STOP'} before resuming.")
        if not status["is_completed"]:
            LOGGER.info("Session budget reached during %s. Rerun to resume.", name)
            break
        if (output / "speed.json").is_file():
            continue
        if session_deadline - time.monotonic() < 30 * 60:
            LOGGER.info("Too little time to finalize %s. Rerun to finish it.", name)
            break
        model, _ = load_final_rt_detr_model(output, DEVICE)
        write_json(
            output / "speed.json",
            measure_inference_speed(
                model, IMAGE_ROOT, validation, objects, ignored, settings["evaluation"], DEVICE
            ),
        )
        if settings["export"]["should_export_test_predictions"]:
            export_test_submission(
                model, IMAGE_ROOT, frames, output, settings["evaluation"], DEVICE
            )
        del model
        gc.collect()
        torch.cuda.empty_cache()
        publish_snapshot(output)
        sync_checkpoint_to_drive_api(api, output, run_folder)
        sync_results_to_drive_api(api, output, run_folder)
    if statuses:
        display(
            pl.from_dicts(
                [
                    {
                        "Run": experiment_label(name),
                        "Epochs completed": status["epochs_completed"],
                        "Updates": status["global_step"],
                        "Image exposures": status["image_exposures"],
                        "Completed": status["is_completed"],
                    }
                    for name, status in statuses.items()
                ]
            )
        )

## 7. Comparison analysis

After training, the analysis stage downloads the committed records of every RT-DETR
experiment and, read-only, the final validation records of Faster R-CNN B1 and the two
stride-five RetinaNet runs. It then measures, from saved predictions only, mAP by object
size and by scene density, same-class near-duplicate pairs, and the RT-DETR mAP after an
added class-wise NMS. It writes `_analysis/analysis.json` and `rt_detr_records.zip`, which
holds the small records for the results notebook, and uploads both to Drive.

In [ ]:
REFERENCES: list[dict] = [
    {
        "key": "faster_rcnn_b1",
        "label": "Faster R-CNN B1 (every fifth frame)",
        "family": "Faster R-CNN",
        "batch": "faster_rcnn_stride5_5epochs_seed2026",
        "folders": ["baseline_stride5"],
        "predictions_file": "val_predictions.json",
    },
    {
        "key": "retinanet_standard_stride5",
        "label": "RetinaNet, standard anchors (every fifth frame)",
        "family": "RetinaNet",
        "batch": "retinanet_stride5_full_5epochs_seed2026",
        "folders": ["standard_anchors_stride5", "R0-S"],
        "predictions_file": "predictions.json",
    },
    {
        "key": "retinanet_smaller_stride5",
        "label": "RetinaNet, smaller anchors (every fifth frame)",
        "family": "RetinaNet",
        "batch": "retinanet_stride5_full_5epochs_seed2026",
        "folders": ["smaller_anchors_stride5", "R1-S"],
        "predictions_file": "predictions.json",
    },
]
if STAGE == "analysis":
    REFERENCE_ROOT = Path("/kaggle/working/references")
    for name in ALL_EXPERIMENTS:
        folder = find_drive_folder(api, batch_folder, [name])
        if folder is not None:
            LOGGER.info(
                "%s: %s",
                name,
                download_committed_files(api, folder, RECORD_FILES, LOCAL_BATCH / name),
            )
    probe_folder = find_drive_folder(api, batch_folder, ["_timing_probe"])
    if probe_folder is not None and "probe.json" in api.children(probe_folder):
        api.download(api.children(probe_folder)["probe.json"]["id"], PROBE_DIRECTORY / "probe.json")
    for reference in REFERENCES:
        for folder_name in reference["folders"]:
            folder = find_drive_folder(
                api, drive_root, ["batches", reference["batch"], folder_name]
            )
            if folder is not None:
                names = (
                    "status.json",
                    "config.json",
                    "metrics.json",
                    reference["predictions_file"],
                )
                LOGGER.info(
                    "%s: %s",
                    reference["key"],
                    download_committed_files(api, folder, names, REFERENCE_ROOT / reference["key"]),
                )
                break
    detectors = [
        {
            "key": name,
            "label": experiment_label(name),
            "family": "RT-DETRv2",
            "directory": str(LOCAL_BATCH / name),
            "predictions_file": "val_predictions.json",
            "is_query_based": True,
        }
        for name in ALL_EXPERIMENTS
    ] + [
        {
            "key": reference["key"],
            "label": reference["label"],
            "family": reference["family"],
            "directory": str(REFERENCE_ROOT / reference["key"]),
            "predictions_file": reference["predictions_file"],
            "is_query_based": False,
        }
        for reference in REFERENCES
    ]
    analysis = run_comparison_analysis(
        SOURCE_ROOT / "val/annotations_coco.json", detectors, ANALYSIS_DIRECTORY
    )
    records_archive = Path("/kaggle/working/rt_detr_records.zip")
    with ZipFile(records_archive, "w", ZIP_DEFLATED) as archive:
        for path in sorted(LOCAL_BATCH.rglob("*")):
            relative = path.relative_to(LOCAL_BATCH)
            if (
                path.is_file()
                and path.suffix in {".json", ".log"}
                and path.name != "val_predictions.json"
                and "sync" not in relative.parts
            ):
                archive.write(path, path.relative_to(Path("/kaggle/working")).as_posix())
    analysis_folder = api.folder(batch_folder, "_analysis")
    api.upload(ANALYSIS_DIRECTORY / "analysis.json", analysis_folder)
    api.upload(records_archive, analysis_folder)
    display(
        pl.from_dicts(
            [
                {
                    "Detector": record["label"],
                    "Status": record["status"],
                    "Final mAP": record["convergence"].get("final_map"),
                    "Predictions analyzed": "predictions" in record,
                }
                for record in analysis["detectors"]
            ]
        )
    )

## 8. Saved progress

We list the saved state of every experiment. After a runtime reset we rerun the stage
`"full"` with the same batch identifier and configuration; the last verified Drive snapshot
is restored and completed epochs are kept. The comparison, curves and analyses are shown in
[the results notebook](../notebooks/05_rt_detr_results.ipynb) after extracting
`rt_detr_records.zip` in the repository root.

In [ ]:
pl.from_dicts(
    [
        {
            "Run": experiment_label(name),
            "Status": summary["status"],
            "Epochs completed": summary.get("epochs_completed", 0),
            "Updates": summary.get("global_step", 0),
            "Training minutes": round(summary.get("wall_train_seconds", 0.0) / 60, 1),
        }
        for name in ALL_EXPERIMENTS
        for summary in [summarize_experiment(LOCAL_BATCH / name)]
    ]
)